# Fast Stage 3 evaluation from completed forecasts

This notebook reuses completed forecast files rather than refitting models. It compares available return-only baselines, HAR/HARX, and the full, VIX-only, and DXY-plus-VIX lasso/ridge/HARX variants on their common weekly origins. The primary table contains only models with forecasts on all shared origins. Previously completed Stage 1 500-origin metrics are shown separately because they cover a different sample. No model-selection decision is implied.

In [1]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

cwd = Path.cwd()
repo_root = next(
    (candidate for candidate in (cwd, *cwd.parents)
     if (candidate / 'pyproject.toml').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository root.')
sys.path.insert(0, str(repo_root / 'src'))

from gold_vol_pipeline.evaluation import qlike, summary_table
from gold_vol_pipeline.regression_models import PREDICTOR_COLUMNS

data_path = repo_root / 'data/processed/stage2_multivariate.csv'
manifest_path = repo_root / 'data/processed/stage2_multivariate_manifest.json'
if not data_path.is_file():
    raise FileNotFoundError('Run the Stage 2 predictor pipeline before this notebook.')
data = pd.read_csv(data_path, parse_dates=['date', 'target_end_date'])
manifest = json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.is_file() else {}
horizon = int(manifest.get('horizon', 5))
if manifest.get('target', 'realized_variance') != 'realized_variance':
    raise ValueError('This evaluation requires the realized_variance target.')
required = {'date', 'log_return', 'realized_variance', 'target_end_date', *PREDICTOR_COLUMNS}
missing = sorted(required - set(data.columns))
if missing:
    raise ValueError(f'Stage 2 data is missing required columns: {missing}')
data = data.sort_values('date').reset_index(drop=True)
print(f'Input rows: {len(data):,}; dates: {data.date.min().date()} through {data.date.max().date()}')
print(f'Target: {horizon}-session forward realized variance; forecasts will be reused, not refitted.')

Input rows: 2,976; dates: 2015-01-05 through 2026-11-03
Target: 5-session forward realized variance; forecasts will be reused, not refitted.


## Reuse completed forecast files

Repeated ARIMA, VAR, GARCH-extension, and rolling cross-validation fits are excluded from this short run. Completed model forecasts are loaded from the tracked output files. The earlier GARCH(1,1) and 500-origin results remain available as separate-sample context, not as part of this common-origin ranking.

In [2]:
forecast_dir = repo_root / 'data/processed/evaluation_forecasts_by_model'
variant_path = repo_root / 'data/processed/predictor_set_variant_forecasts.csv'
required_columns = {'origin_date', 'model', 'forecast', 'actual', 'converged'}
forecast_frames = []
source_files = []
for path in sorted(forecast_dir.glob('*.csv')):
    if path.stem == 'harx_vix':
        continue
    frame = pd.read_csv(path, parse_dates=['origin_date'])
    missing = required_columns - set(frame.columns)
    if missing:
        raise ValueError(f'{path} is missing columns: {sorted(missing)}')
    forecast_frames.append(frame)
    source_files.append(str(path.relative_to(repo_root)))
variants = pd.read_csv(variant_path, parse_dates=['origin_date'])
missing = required_columns - set(variants.columns)
if missing:
    raise ValueError(f'{variant_path} is missing columns: {sorted(missing)}')
forecast_frames.append(variants)
source_files.append(str(variant_path.relative_to(repo_root)))
results = pd.concat(forecast_frames, ignore_index=True)
results = results.loc[results['converged'].astype(bool)].copy()
if results.duplicated(['origin_date', 'model']).any():
    duplicates = results.loc[results.duplicated(['origin_date', 'model'], keep=False), ['origin_date', 'model']]
    raise ValueError(f'Duplicate forecast/model rows found: {duplicates.head().to_dict("records")}')
origin_sets = results.groupby('model')['origin_date'].apply(set)
common_origins = set.intersection(*origin_sets.tolist())
results = results.loc[results['origin_date'].isin(common_origins)].sort_values(['origin_date', 'model']).reset_index(drop=True)
if results.empty or results.groupby('model').size().nunique() != 1:
    raise ValueError('Saved forecasts do not share a complete common-origin sample.')
actual_counts = results.groupby('origin_date')['actual'].nunique()
if (actual_counts != 1).any():
    raise ValueError('Saved model outputs disagree on actual target values.')
forecast_path = repo_root / 'data/processed/evaluation_forecasts.csv'
results.to_csv(forecast_path, index=False)
manifest = {
    'evaluation': 'reused completed forecasts; no model refits',
    'target': 'five-session forward realized variance',
    'horizon': horizon,
    'common_origin_count': len(common_origins),
    'origin_range': {'start': results.origin_date.min().date().isoformat(), 'end': results.origin_date.max().date().isoformat()},
    'models': sorted(results.model.unique().tolist()),
    'source_files': source_files,
    'excluded_slow_fits': ['auto_arima', 'var', 'gjr_garch', 'egarch_asym', 'garch_t', 'garchx'],
    'excluded_due_to_sample_mismatch': ['garch11 500-origin forecasts'],
}
(forecast_path.with_name('evaluation_forecasts_manifest.json')).write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')
prior_metrics_path = repo_root / 'cross_model_comparison_with_baselines.csv'
if prior_metrics_path.is_file():
    prior_metrics = pd.read_csv(prior_metrics_path)
    print('Previously completed cross-model metrics (different model-specific sample sizes; shown for context only):')
    display(prior_metrics)
print(f'Reused {len(results):,} saved forecast rows across {results.model.nunique()} models and {len(common_origins)} shared origins.')
print(f'Wrote common-origin forecasts to {forecast_path}')

Previously completed cross-model metrics (different model-specific sample sizes; shown for context only):


,model,n,MSE,RMSE,MAE,QLIKE,mean_forecast,mean_actual,convergence_rate
0,lasso_vix,445,9.847662e-07,0.000992,0.000429,0.456534,0.000546,0.000625,1.0
1,ridge_vix,445,9.897790e-07,0.000995,0.000430,0.456570,0.000552,0.000625,1.0
2,harx_vix,445,9.900030e-07,0.000995,0.000431,0.456025,0.000555,0.000625,1.0
3,lasso_full,445,1.007406e-06,0.001004,0.000422,0.472836,0.000485,0.000625,1.0
4,harx_dxy_vix,445,1.015396e-06,0.001008,0.000436,0.458604,0.000563,0.000625,1.0
5,lasso_dxy_vix,445,1.016488e-06,0.001008,0.000433,0.463055,0.000548,0.000625,1.0
6,ridge_dxy_vix,445,1.034803e-06,0.001017,0.000436,0.461836,0.000558,0.000625,1.0
7,ridge_full,445,1.115971e-06,0.001056,0.000437,0.448565,0.000542,0.000625,1.0
8,harx_full,445,1.276503e-06,0.001130,0.000457,0.448658,0.000570,0.000625,1.0
9,garch11,488,3.575084e-06,0.001891,0.000783,0.577452,0.000760,0.001232,1.0


Reused 7,565 saved forecast rows across 17 models and 445 shared origins.
Wrote common-origin forecasts to /Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject/data/processed/evaluation_forecasts.csv


In [3]:
summary = summary_table(results, benchmark='historical_variance', horizon=horizon, top_n=5)
display(summary)
summary_path = repo_root / 'data/processed/evaluation_summary.csv'
summary.to_csv(summary_path, index=False)
print(f'Wrote loss and inference summary to {summary_path}')
dm_mse = summary.attrs['dm_pvalue_matrix_mse']
dm_qlike = summary.attrs['dm_pvalue_matrix_qlike']
display(dm_qlike.style.format('{:.3g}'))

,model,n,MSE,MAE,QLIKE,MSE_ratio_to_benchmark,QLIKE_ratio_to_benchmark,DM_MSE_p_raw,DM_MSE_p_holm,DM_QLIKE_p_raw,DM_QLIKE_p_holm,MZ_a,MZ_b,MZ_wald_p
0,ewma,445,9.759453e-07,0.000442,0.450875,0.865795,0.709357,0.123425,1.000000,0.024201,0.620403,0.000172,0.743453,6.446732e-03
1,har,445,9.813509e-07,0.000416,0.464897,0.870590,0.731418,0.045193,0.940909,0.021026,0.620403,0.000017,1.161917,9.950482e-02
2,lasso_vix,445,9.847662e-07,0.000429,0.456534,0.873620,0.718260,0.150020,1.000000,0.019388,0.620403,0.000247,0.692422,3.194311e-03
3,ewma_fixed,445,9.869340e-07,0.000452,0.454959,0.875543,0.715782,0.161166,1.000000,0.044805,0.940909,0.000222,0.653409,3.081725e-06
4,ridge_vix,445,9.897790e-07,0.000430,0.456570,0.878067,0.718317,0.153923,1.000000,0.019545,0.620403,0.000258,0.665262,7.823178e-04
5,harx_vix,445,9.900030e-07,0.000431,0.456025,0.878266,0.717459,0.154392,1.000000,0.020075,0.620403,0.000259,0.661058,5.163545e-04
6,lasso_full,445,1.007406e-06,0.000422,0.472836,0.893705,0.743908,0.196010,1.000000,0.022146,0.620403,0.000224,0.826766,1.041483e-02
7,harx_dxy_vix,445,1.015396e-06,0.000436,0.458604,0.900793,0.721517,0.199870,1.000000,0.020696,0.620403,0.000280,0.613472,5.138237e-05
8,lasso_dxy_vix,445,1.016488e-06,0.000433,0.463055,0.901761,0.728520,0.216624,1.000000,0.020742,0.620403,0.000281,0.628862,5.913512e-04
9,ridge_dxy_vix,445,1.034803e-06,0.000436,0.461836,0.918009,0.726603,0.325943,1.000000,0.021773,0.620403,0.000290,0.600746,9.829798e-04


Wrote loss and inference summary to /Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject/data/processed/evaluation_summary.csv


,ewma,har,lasso_vix,ewma_fixed,ridge_vix
ewma,1,0.37,0.871,0.799,0.872
har,0.37,1,0.781,0.628,0.785
lasso_vix,0.871,0.781,1,0.961,0.981
ewma_fixed,0.799,0.628,0.961,1,0.961
ridge_vix,0.872,0.785,0.981,0.961,1


## Full versus reduced predictor sets

Compare full seven-variable models against a VIX-only set (the Yahoo/FRED screen's significant predictor) and a DXY-plus-VIX set (both individually significant in the separate LSEG screen). DXY-plus-VIX is a cross-source sensitivity set, not a significance claim for the Yahoo/FRED sample. Each model family uses identical forecast origins and its full-set version as the Diebold-Mariano benchmark; report loss ratios and Holm-adjusted MSE/QLIKE p-values within each family. These comparisons evaluate predictive performance rather than selecting variables by coefficient p-values alone.

In [4]:
predictor_comparisons = {
    'HARX': ['harx_full', 'harx_dxy_vix', 'harx_vix'],
    'Lasso': ['lasso_full', 'lasso_dxy_vix', 'lasso_vix'],
    'Ridge': ['ridge_full', 'ridge_dxy_vix', 'ridge_vix'],
}
comparison_tables = []
for family, model_names in predictor_comparisons.items():
    family_results = results.loc[results['model'].isin(model_names)].copy()
    family_summary = summary_table(
        family_results,
        benchmark=model_names[0],
        horizon=horizon,
        top_n=len(model_names),
    )
    family_summary.insert(0, 'family', family)
    family_summary.attrs = {}
    comparison_tables.append(family_summary)
predictor_comparison = pd.concat(comparison_tables, ignore_index=True)
comparison_columns = [
    'family', 'model', 'n', 'MSE', 'MAE', 'QLIKE',
    'MSE_ratio_to_benchmark', 'QLIKE_ratio_to_benchmark',
    'DM_MSE_p_holm', 'DM_QLIKE_p_holm',
]
display(predictor_comparison[comparison_columns].style.format({
    'MSE': '{:.6g}',
    'MAE': '{:.6g}',
    'QLIKE': '{:.4f}',
    'MSE_ratio_to_benchmark': '{:.3f}',
    'QLIKE_ratio_to_benchmark': '{:.3f}',
    'DM_MSE_p_holm': '{:.4g}',
    'DM_QLIKE_p_holm': '{:.4g}',
}).hide(axis='index'))
comparison_path = repo_root / 'data/processed/predictor_set_comparison.csv'
predictor_comparison[comparison_columns].to_csv(comparison_path, index=False)
print(f'Wrote predictor-set comparison to {comparison_path}')

family,model,n,MSE,MAE,QLIKE,MSE_ratio_to_benchmark,QLIKE_ratio_to_benchmark,DM_MSE_p_holm,DM_QLIKE_p_holm
HARX,harx_vix,445,9.90003e-07,0.00043069,0.4560,0.776,1.016,1,1
HARX,harx_dxy_vix,445,1.0154e-06,0.000436203,0.4586,0.795,1.022,1,1
HARX,harx_full,445,1.2765e-06,0.000457265,0.4487,1.000,1.000,1,1
Lasso,lasso_vix,445,9.84766e-07,0.000428678,0.4565,0.978,0.966,0.8952,0.7356
Lasso,lasso_full,445,1.00741e-06,0.000421803,0.4728,1.000,1.000,1,1
Lasso,lasso_dxy_vix,445,1.01649e-06,0.000433093,0.4631,1.009,0.979,0.8952,0.8952
Ridge,ridge_vix,445,9.89779e-07,0.000429694,0.4566,0.887,1.018,0.862,1
Ridge,ridge_dxy_vix,445,1.0348e-06,0.00043636,0.4618,0.927,1.030,0.9231,1
Ridge,ridge_full,445,1.11597e-06,0.000436695,0.4486,1.000,1.000,1,1


Wrote predictor-set comparison to /Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject/data/processed/predictor_set_comparison.csv


In [5]:
plot_dir = repo_root / 'plots/final_evaluation'
plot_dir.mkdir(parents=True, exist_ok=True)
valid_summary = summary.loc[summary['n'] > 0]
best_models = valid_summary['model'].head(3).tolist()
benchmark = 'historical_variance'
plot_models = list(dict.fromkeys([*best_models, benchmark]))

# Actual versus forecast for the three lowest-MSE models and the historical-variance benchmark.
wide = results[results.model.isin(plot_models)].pivot(index='origin_date', columns='model', values='forecast')
actual = results[results.model == benchmark].set_index('origin_date')['actual']
figure, axis = plt.subplots(figsize=(13, 6))
axis.plot(actual.index, actual, label='Observed', color='black', linewidth=1.5)
for name in plot_models:
    if name in wide:
        axis.plot(wide.index, wide[name], label=name, alpha=0.85)
axis.set_title('5-session forward realized variance, observed vs forecast')
axis.set_xlabel('Forecast origin date')
axis.set_ylabel('Realized variance (squared log-return units; 5 sessions)')
axis.legend(ncol=2)
figure.tight_layout()
figure.savefig(plot_dir / 'actual_vs_forecast.png', dpi=200)
plt.close(figure)

# Cumulative QLIKE loss difference: negative values favor the candidate.
benchmark_rows = results[results.model == benchmark][['origin_date', 'actual', 'forecast']].rename(columns={'forecast': 'benchmark'})
figure, axis = plt.subplots(figsize=(13, 6))
for name in plot_models:
    if name == benchmark:
        continue
    model_rows = results[results.model == name][['origin_date', 'actual', 'forecast']]
    aligned = model_rows.merge(benchmark_rows, on=['origin_date', 'actual'], validate='one_to_one').dropna()
    if len(aligned) and (aligned[['actual', 'forecast', 'benchmark']] > 0).all().all():
        model_loss, _ = qlike(aligned.actual, aligned.forecast)
        benchmark_loss, _ = qlike(aligned.actual, aligned.benchmark)
        axis.plot(aligned.origin_date, (model_loss - benchmark_loss).cumsum(), label=name)
axis.axhline(0, color='black', linewidth=0.8)
axis.set_title(f'Cumulative QLIKE difference vs {benchmark}')
axis.set_xlabel('Forecast origin date')
axis.set_ylabel('Cumulative QLIKE loss difference (candidate − benchmark; 5 sessions)')
axis.legend()
figure.tight_layout()
figure.savefig(plot_dir / 'cumulative_qlike_vs_historical_variance.png', dpi=200)
plt.close(figure)

# Selected lasso coefficients across origins, with selection frequency.
lasso_rows = results[(results.model == 'lasso_full') & results.coefficients.notna()].copy()
coefficient_rows = []
for row in lasso_rows.itertuples():
    for feature, coefficient in json.loads(row.coefficients).items():
        coefficient_rows.append({'origin_date': row.origin_date, 'feature': feature, 'coefficient': coefficient})
coefficient_frame = pd.DataFrame(coefficient_rows)
figure, axes = plt.subplots(2, 1, figsize=(13, 9), gridspec_kw={'height_ratios': [2, 1]})
if not coefficient_frame.empty:
    frequencies = coefficient_frame.groupby('feature').origin_date.nunique().div(lasso_rows.origin_date.nunique()).sort_values(ascending=False)
    selected_features = frequencies.head(8).index.tolist()
    for feature in selected_features:
        path = coefficient_frame[coefficient_frame.feature == feature]
        axes[0].plot(path.origin_date, path.coefficient, label=feature)
    axes[0].legend(ncol=2)
    axes[1].bar(frequencies.head(12).index, frequencies.head(12).values)
    axes[1].tick_params(axis='x', rotation=45)
axes[0].set_title('Lasso selected coefficient paths across origins')
axes[0].set_ylabel('Log-variance coefficient (training-standardized predictors; 5 sessions)')
axes[0].set_xlabel('Forecast origin date')
axes[1].set_title('Lasso coefficient selection frequency (5-session target)')
axes[1].set_ylabel('Fraction of fitted origins')
axes[1].set_xlabel('Feature')
figure.tight_layout()
figure.savefig(plot_dir / 'lasso_coefficients.png', dpi=200)
plt.close(figure)

# Pairwise DM QLIKE p-values for the top models.
figure, axis = plt.subplots(figsize=(8, 7))
image = axis.imshow(dm_qlike.to_numpy(dtype=float), vmin=0, vmax=1, cmap='viridis_r')
axis.set_xticks(range(len(dm_qlike.columns)), dm_qlike.columns, rotation=45, ha='right')
axis.set_yticks(range(len(dm_qlike.index)), dm_qlike.index)
axis.set_title('Pairwise Diebold-Mariano p-values (QLIKE; 5-session horizon)')
figure.colorbar(image, ax=axis, label='Two-sided p-value')
figure.tight_layout()
figure.savefig(plot_dir / 'dm_qlike_heatmap.png', dpi=200)
plt.close(figure)
print(f'Saved 200 dpi figures to {plot_dir}')

Saved 200 dpi figures to /Users/enoch/Documents/Gold-vol-forecasting_ECON4120_GroupProject/plots/final_evaluation


## Model-comparison handoff

The common-origin comparison uses only previously completed forecasts; no model fits were repeated. The 500-origin Stage 1 results in `cross_model_comparison_with_baselines.csv` use a different sample and must not be ranked directly against this table. Auto-ARIMA, VAR, GJR-GARCH, asymmetric EGARCH, Student-t GARCH, and GARCH-X are excluded from the short evaluation run; this is a runtime decision, not a finding that they are unsuitable forecasting methods.

Interpret the loss, Diebold-Mariano, and Mincer-Zarnowitz evidence alongside the overlapping-target dependence and source-timing limitations. This exploratory comparison does not imply a model-selection decision.